# 🏬 Store Sales — Time Series Forecasting

**Kaggle Competition**: Predict grocery store sales for Corporación Favorita, a large Ecuadorian-based grocery retailer.

**Objective**: Forecast 16 days of sales (Aug 16–31, 2017) for 54 stores across 33 product families (1,782 individual time series).

**Evaluation Metric**: Root Mean Squared Logarithmic Error (RMSLE)

$$RMSLE = \sqrt{\frac{1}{n}\sum_{i=1}^{n}\left(\log(1+\hat{y}_i) - \log(1+y_i)\right)^2}$$

**Approach**: LightGBM with extensive feature engineering. We apply `log1p()` to the target variable so that optimizing standard RMSE is mathematically equivalent to optimizing RMSLE.

---

## 📦 1. Import Libraries

In [1]:
# Core data manipulation and visualization libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Machine learning libraries
import lightgbm as lgb
from sklearn.metrics import mean_squared_error

# Global configuration
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')

print("All libraries imported successfully!")

All libraries imported successfully!


In [ ]:
# Detect the correct path to the Dataset folder
# Works regardless of where Jupyter sets the working directory
import os

# Strategy: search for the Dataset folder relative to common locations
_candidates = [
    'Dataset/',                        # If cwd is project root
    '../Dataset/',                     # If cwd is Model/
    os.path.join(os.path.dirname(os.path.abspath('__file__')), '..', 'Dataset'),  # Relative to notebook
]

DATA_DIR = None
for path in _candidates:
    if os.path.isdir(path):
        DATA_DIR = path if path.endswith('/') else path + '/'
        break

# Fallback: use absolute path as last resort
if DATA_DIR is None:
    DATA_DIR = r'C:\Users\admin\Documents\Portofolio\Store Sales Forecasting\Dataset/'

print(f'Working directory: {os.getcwd()}')
print(f'Using DATA_DIR:    {os.path.abspath(DATA_DIR)}')

# Load all datasets with automatic date parsing
train = pd.read_csv(f'{DATA_DIR}train.csv', parse_dates=['date'])
test = pd.read_csv(f'{DATA_DIR}test.csv', parse_dates=['date'])
stores = pd.read_csv(f'{DATA_DIR}stores.csv')
oil = pd.read_csv(f'{DATA_DIR}oil.csv', parse_dates=['date'])
holidays = pd.read_csv(f'{DATA_DIR}holidays_events.csv', parse_dates=['date'])
transactions = pd.read_csv(f'{DATA_DIR}transactions.csv', parse_dates=['date'])
sample_submission = pd.read_csv(f'{DATA_DIR}sample_submission.csv')

# Print dataset dimensions and date ranges
print("=" * 70)
print("DATASET OVERVIEW")
print("=" * 70)
print(f"{'Dataset':<15} {'Rows':>12} {'Cols':>6}   Date Range")
print("-" * 70)
print(f"{'Train':<15} {train.shape[0]:>12,} {train.shape[1]:>6}   {train['date'].min().date()} to {train['date'].max().date()}")
print(f"{'Test':<15} {test.shape[0]:>12,} {test.shape[1]:>6}   {test['date'].min().date()} to {test['date'].max().date()}")
print(f"{'Stores':<15} {stores.shape[0]:>12,} {stores.shape[1]:>6}")
print(f"{'Oil':<15} {oil.shape[0]:>12,} {oil.shape[1]:>6}   {oil['date'].min().date()} to {oil['date'].max().date()}")
print(f"{'Holidays':<15} {holidays.shape[0]:>12,} {holidays.shape[1]:>6}")
print(f"{'Transactions':<15} {transactions.shape[0]:>12,} {transactions.shape[1]:>6}")
print(f"{'Submission':<15} {sample_submission.shape[0]:>12,} {sample_submission.shape[1]:>6}")
print(f"\nUnique stores: {train['store_nbr'].nunique()} | Unique product families: {train['family'].nunique()}")
print(f"Total individual time series: {train['store_nbr'].nunique()} x {train['family'].nunique()} = {train['store_nbr'].nunique() * train['family'].nunique():,}")

In [2]:
# Define the data directory
DATA_DIR = 'Dataset/'

# Load all datasets with automatic date parsing
train = pd.read_csv(f'{DATA_DIR}train.csv', parse_dates=['date'])
test = pd.read_csv(f'{DATA_DIR}test.csv', parse_dates=['date'])
stores = pd.read_csv(f'{DATA_DIR}stores.csv')
oil = pd.read_csv(f'{DATA_DIR}oil.csv', parse_dates=['date'])
holidays = pd.read_csv(f'{DATA_DIR}holidays_events.csv', parse_dates=['date'])
transactions = pd.read_csv(f'{DATA_DIR}transactions.csv', parse_dates=['date'])
sample_submission = pd.read_csv(f'{DATA_DIR}sample_submission.csv')

# Print dataset dimensions and date ranges
print("=" * 70)
print("DATASET OVERVIEW")
print("=" * 70)
print(f"{'Dataset':<15} {'Rows':>12} {'Cols':>6}   Date Range")
print("-" * 70)
print(f"{'Train':<15} {train.shape[0]:>12,} {train.shape[1]:>6}   {train['date'].min().date()} to {train['date'].max().date()}")
print(f"{'Test':<15} {test.shape[0]:>12,} {test.shape[1]:>6}   {test['date'].min().date()} to {test['date'].max().date()}")
print(f"{'Stores':<15} {stores.shape[0]:>12,} {stores.shape[1]:>6}")
print(f"{'Oil':<15} {oil.shape[0]:>12,} {oil.shape[1]:>6}   {oil['date'].min().date()} to {oil['date'].max().date()}")
print(f"{'Holidays':<15} {holidays.shape[0]:>12,} {holidays.shape[1]:>6}")
print(f"{'Transactions':<15} {transactions.shape[0]:>12,} {transactions.shape[1]:>6}")
print(f"{'Submission':<15} {sample_submission.shape[0]:>12,} {sample_submission.shape[1]:>6}")
print(f"\nUnique stores: {train['store_nbr'].nunique()} | Unique product families: {train['family'].nunique()}")
print(f"Total individual time series: {train['store_nbr'].nunique()} x {train['family'].nunique()} = {train['store_nbr'].nunique() * train['family'].nunique():,}")

FileNotFoundError: [Errno 2] No such file or directory: 'Dataset/train.csv'

## 🔍 3. Exploratory Data Analysis

Let's explore the data to understand sales patterns, distributions, seasonality, and relationships between features.

In [ ]:
# Display basic statistics about the training data
print("=" * 60)
print("TRAINING DATA STATISTICS")
print("=" * 60)

# Date range and prediction horizon
total_days = (train['date'].max() - train['date'].min()).days
pred_horizon = (test['date'].max() - test['date'].min()).days + 1
print(f"\nTraining period: {train['date'].min().date()} to {train['date'].max().date()} ({total_days} days)")
print(f"Prediction horizon: {pred_horizon} days")

# Sales distribution summary
print(f"\nSales Distribution:")
print(train['sales'].describe().to_string())

# Zero-sales analysis (important for understanding data sparsity)
zero_count = (train['sales'] == 0).sum()
total_count = len(train)
print(f"\nZero-sales records:     {zero_count:>12,} ({zero_count / total_count * 100:.1f}%)")
print(f"Non-zero sales records: {total_count - zero_count:>12,} ({(total_count - zero_count) / total_count * 100:.1f}%)")

# Missing values across all datasets
print(f"\n{'=' * 60}")
print("MISSING VALUES")
print("=" * 60)
for name, df_check in [('Train', train), ('Test', test), ('Oil', oil), ('Holidays', holidays)]:
    missing = df_check.isnull().sum()
    if missing.sum() > 0:
        print(f"\n{name}:")
        print(missing[missing > 0].to_string())
    else:
        print(f"\n{name}: No missing values")

### 3.1 Sales Patterns Overview

In [ ]:
# Aggregate daily total sales across all stores and families
daily_sales = train.groupby('date')['sales'].sum()

fig, axes = plt.subplots(2, 2, figsize=(18, 10))

# ---- Plot 1: Overall sales trend over time ----
axes[0, 0].plot(daily_sales.index, daily_sales.values,
                linewidth=0.5, alpha=0.8, color='#2196F3')
axes[0, 0].set_title('Total Daily Sales Over Time', fontsize=13, fontweight='bold')
axes[0, 0].set_xlabel('Date')
axes[0, 0].set_ylabel('Total Sales')

# ---- Plot 2: Average sales by day of week ----
dow_sales = train.groupby(train['date'].dt.day_name())['sales'].mean()
dow_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dow_sales = dow_sales.reindex(dow_order)
colors_dow = ['#42A5F5'] * 5 + ['#EF5350', '#EF5350']  # Highlight weekends in red
axes[0, 1].bar(range(7), dow_sales.values, color=colors_dow, edgecolor='white')
axes[0, 1].set_xticks(range(7))
axes[0, 1].set_xticklabels([d[:3] for d in dow_order])
axes[0, 1].set_title('Average Sales by Day of Week', fontsize=13, fontweight='bold')
axes[0, 1].set_ylabel('Average Sales')

# ---- Plot 3: Distribution of log-transformed sales ----
non_zero_sales = train.loc[train['sales'] > 0, 'sales']
axes[1, 0].hist(np.log1p(non_zero_sales), bins=80,
                color='#66BB6A', edgecolor='white', alpha=0.8)
axes[1, 0].set_title('Distribution of log1p(Sales) [Non-Zero Only]', fontsize=13, fontweight='bold')
axes[1, 0].set_xlabel('log1p(Sales)')
axes[1, 0].set_ylabel('Frequency')

# ---- Plot 4: Monthly seasonality pattern ----
monthly_sales = train.groupby(train['date'].dt.month)['sales'].mean()
month_labels = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
axes[1, 1].bar(range(1, 13), monthly_sales.values, color='#FFA726', edgecolor='white')
axes[1, 1].set_xticks(range(1, 13))
axes[1, 1].set_xticklabels(month_labels)
axes[1, 1].set_title('Average Sales by Month', fontsize=13, fontweight='bold')
axes[1, 1].set_ylabel('Average Sales')

plt.suptitle('Sales Patterns Overview', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 3.2 Sales by Product Family

In [ ]:
# Rank product families by total sales volume
family_sales = train.groupby('family')['sales'].sum().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(12, 10))
colors_bar = plt.cm.viridis(np.linspace(0.2, 0.8, len(family_sales)))
ax.barh(range(len(family_sales)), family_sales.values, color=colors_bar)
ax.set_yticks(range(len(family_sales)))
ax.set_yticklabels(family_sales.index, fontsize=9)
ax.set_xlabel('Total Sales', fontsize=12)
ax.set_title('Total Sales by Product Family', fontsize=14, fontweight='bold')

# Format x-axis to show millions
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
plt.tight_layout()
plt.show()

# Show top 5 and bottom 5 families
print("Top 5 Product Families by Total Sales:")
for i, (fam, sales) in enumerate(family_sales.tail(5).iloc[::-1].items(), 1):
    print(f"  {i}. {fam:<25s} {sales:>15,.0f}")

print("\nBottom 5 Product Families by Total Sales:")
for i, (fam, sales) in enumerate(family_sales.head(5).items(), 1):
    print(f"  {i}. {fam:<25s} {sales:>15,.0f}")

### 3.3 Oil Prices and Store Transactions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# ---- Plot 1: Oil price trend ----
axes[0].plot(oil['date'], oil['dcoilwtico'], color='#FF7043', linewidth=1)
axes[0].set_title('Daily Oil Price (WTI Crude)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Price (USD)')
oil_mean = oil['dcoilwtico'].mean()
axes[0].axhline(y=oil_mean, color='gray', linestyle='--', alpha=0.5,
                label=f'Mean: ${oil_mean:.1f}')
axes[0].legend()

# ---- Plot 2: Average transactions by store type ----
store_trans = transactions.merge(stores, on='store_nbr')
type_trans = store_trans.groupby('type')['transactions'].mean().sort_values()
type_colors = ['#26A69A', '#42A5F5', '#7E57C2', '#EF5350', '#FFA726']
axes[1].barh(type_trans.index, type_trans.values, color=type_colors[:len(type_trans)])
axes[1].set_title('Average Transactions by Store Type', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Average Daily Transactions')

plt.tight_layout()
plt.show()

## 🔧 4. Data Preprocessing

Key preprocessing steps:
1. **Oil prices**: Fill missing values using linear interpolation
2. **Holidays**: Create binary flags for national holidays
3. **Merge**: Combine all auxiliary datasets with train/test into a unified DataFrame
4. **Target transformation**: Apply `log1p()` so that RMSE optimization ≡ RMSLE optimization

In [ ]:
# ============================================================
# 4.1 Preprocess Oil Prices
# ============================================================

# Resample to daily frequency to fill date gaps, then interpolate
oil = oil.set_index('date').resample('D').last().reset_index()
oil['dcoilwtico'] = oil['dcoilwtico'].interpolate(method='linear')
oil['dcoilwtico'] = oil['dcoilwtico'].ffill().bfill()  # Handle edges

print(f"Oil: {oil['dcoilwtico'].isnull().sum()} missing values remaining")

# ============================================================
# 4.2 Preprocess Holidays
# ============================================================

# Extract national holidays (most impactful on sales across all stores)
national_holidays = holidays[
    (holidays['locale'] == 'National') &
    (holidays['transferred'] == False)
][['date']].drop_duplicates()
national_holidays['is_national_holiday'] = 1

print(f"National holidays: {len(national_holidays)} unique dates")

# ============================================================
# 4.3 Combine Train and Test for unified feature engineering
# ============================================================

# Tag rows so we can split them back later
train['is_train'] = True
test['is_train'] = False

# Concatenate train and test into a single DataFrame
df = pd.concat([train, test], sort=False).reset_index(drop=True)
print(f"\nCombined DataFrame: {df.shape[0]:,} rows x {df.shape[1]} cols")

# Merge store metadata (city, state, store type, cluster)
df = df.merge(stores, on='store_nbr', how='left')

# Merge oil prices
df = df.merge(oil[['date', 'dcoilwtico']], on='date', how='left')
df['dcoilwtico'] = df['dcoilwtico'].ffill().bfill()

# Merge national holidays
df = df.merge(national_holidays, on='date', how='left')
df['is_national_holiday'] = df['is_national_holiday'].fillna(0).astype(int)

# ============================================================
# 4.4 Apply log1p transformation to the target variable
# ============================================================
# Key insight: minimizing RMSE on log1p(sales) is equivalent to
# minimizing RMSLE on the original sales values
df['sales_log1p'] = np.log1p(df['sales'])

print(f"\nPreprocessing complete!")
print(f"Final shape: {df.shape[0]:,} rows x {df.shape[1]} cols")

## ⚙️ 5. Feature Engineering

This is the **most critical step** for achieving competitive performance. We engineer features across several categories:

| Category | Description | Examples |
|----------|-------------|---------|
| Calendar | Time-based patterns and seasonality | Day of week, month, payday indicators |
| Lag | Historical sales values (min lag ≥ 16) | Sales 16, 21, 28 days ago |
| Rolling | Smoothed historical statistics | 7-day and 14-day moving averages |
| Promotion | Current and historical promotion status | Promotion lags, rolling promo rate |
| External | Oil prices and holiday effects | Oil price lags, national holiday flag |
| Store | Store-level metadata | Store type, cluster, city |

> **Important**: The minimum lag is 16 days because our forecast horizon is 16 days (Aug 16–31). Using shorter lags would cause **data leakage** since those values are unknown at prediction time.

In [ ]:
# Sort data by (store, family, date) for correct lag/rolling computation
df = df.sort_values(['store_nbr', 'family', 'date']).reset_index(drop=True)

print("Creating features...")

# ============================================================
# 5.1 Calendar Features
# ============================================================

# Basic date components
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day
df['dayofweek'] = df['date'].dt.dayofweek       # 0=Monday, 6=Sunday
df['dayofyear'] = df['date'].dt.dayofyear
df['weekofyear'] = df['date'].dt.isocalendar().week.astype(int)
df['quarter'] = df['date'].dt.quarter

# Binary indicators for special calendar positions
df['is_weekend'] = (df['dayofweek'] >= 5).astype(int)
df['is_month_start'] = df['date'].dt.is_month_start.astype(int)
df['is_month_end'] = df['date'].dt.is_month_end.astype(int)

# Ecuador payday effect: salaries are typically paid on the 15th
# and the last day of each month ("Quincena")
df['is_payday'] = (
    df['day'].isin([15, 16]) | df['date'].dt.is_month_end
).astype(int)

# Cyclical encoding of periodic features using sine/cosine transforms
# This preserves the circular nature (e.g., Dec is close to Jan)
df['dow_sin'] = np.sin(2 * np.pi * df['dayofweek'] / 7)
df['dow_cos'] = np.cos(2 * np.pi * df['dayofweek'] / 7)
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
df['day_sin'] = np.sin(2 * np.pi * df['dayofyear'] / 365.25)
df['day_cos'] = np.cos(2 * np.pi * df['dayofyear'] / 365.25)

print("  Calendar features created")

# ============================================================
# 5.2 Lag Features (minimum lag = 16 to prevent data leakage)
# ============================================================

# Group by each individual time series (one per store-family pair)
grouped = df.groupby(['store_nbr', 'family'])

# Sales lag features — capture recent and periodic historical patterns
lag_days = [16, 17, 18, 19, 20, 21, 28, 35, 42, 49]
for lag in lag_days:
    df[f'sales_lag_{lag}'] = grouped['sales_log1p'].shift(lag)

# Promotion lag features
for lag in [16, 21, 28]:
    df[f'promo_lag_{lag}'] = grouped['onpromotion'].shift(lag)

print(f"  Lag features created ({len(lag_days)} sales lags + 3 promo lags)")

# ============================================================
# 5.3 Rolling Window Features
# ============================================================

# Rolling statistics computed ON the lag-16 column.
# Since lag-16 already shifts data by 16 positions, a rolling(W)
# window where W <= 16 stays safely within the same group boundary.
for window in [7, 14]:
    df[f'sales_roll_mean_{window}'] = (
        df['sales_lag_16'].rolling(window, min_periods=1).mean()
    )
    df[f'sales_roll_std_{window}'] = (
        df['sales_lag_16'].rolling(window, min_periods=1).std()
    )

# For a larger rolling window (28), we use lag-28 as the base
# to maintain group boundary safety (rolling window <= shift amount)
df['sales_roll_mean_28'] = (
    df.groupby(['store_nbr', 'family'])['sales_log1p']
    .shift(28)
    .rolling(28, min_periods=1)
    .mean()
)

# Min and max over recent lag period
df['sales_roll_min_7'] = df['sales_lag_16'].rolling(7, min_periods=1).min()
df['sales_roll_max_7'] = df['sales_lag_16'].rolling(7, min_periods=1).max()

# Rolling promotion statistics
df['promo_roll_mean_7'] = (
    df['promo_lag_16'].rolling(7, min_periods=1).mean()
)
df['promo_roll_mean_14'] = (
    df['promo_lag_16'].rolling(14, min_periods=1).mean()
)

print("  Rolling window features created")

# ============================================================
# 5.4 Encode Categorical Features (Label Encoding)
# ============================================================

# Label encode categorical variables for LightGBM
df['family_code'] = df['family'].astype('category').cat.codes
df['city_code'] = df['city'].astype('category').cat.codes
df['state_code'] = df['state'].astype('category').cat.codes
df['type_code'] = df['type'].astype('category').cat.codes

print("  Categorical features encoded")

# ============================================================
# 5.5 Oil Price Features
# ============================================================

# Lagged oil prices (global shift, not per-group)
df['oil_lag_1'] = df['dcoilwtico'].shift(1)
df['oil_lag_7'] = df['dcoilwtico'].shift(7)

# Smoothed oil price trends
df['oil_roll_mean_7'] = df['dcoilwtico'].rolling(7, min_periods=1).mean()
df['oil_roll_mean_28'] = df['dcoilwtico'].rolling(28, min_periods=1).mean()

print("  Oil price features created")

# ============================================================
# Summary
# ============================================================
print(f"\nFeature engineering complete!")
print(f"Total columns: {df.shape[1]}")

## 🤖 6. Model Training — LightGBM

**Training Strategy**:
- **Validation Split**: Use the last 16 days of training data (Aug 1–15, 2017) as the validation set, mirroring the 16-day test horizon
- **Target**: `log1p(sales)` — so standard RMSE loss directly minimizes RMSLE
- **Model**: LightGBM (Gradient Boosted Decision Trees) with early stopping
- **Evaluation**: Monitor RMSE on validation set during training

In [ ]:
# ============================================================
# 6.1 Define Feature Columns
# ============================================================

# Columns to exclude from model features (identifiers, targets, metadata)
exclude_cols = [
    'id', 'date', 'sales', 'sales_log1p', 'is_train',
    'family', 'city', 'state', 'type'
]

# All remaining columns become input features
feature_cols = [col for col in df.columns if col not in exclude_cols]
print(f"Number of features: {len(feature_cols)}")
print(f"\nFeature list:")
for i, col in enumerate(feature_cols, 1):
    print(f"  {i:>2}. {col}")

# ============================================================
# 6.2 Create Train / Validation / Test Splits
# ============================================================

# Validation period: last 16 days of training data
# This simulates the exact same forecast scenario as the test set
VAL_START = pd.Timestamp('2017-08-01')
VAL_END = pd.Timestamp('2017-08-15')

# Training: all data before validation period, excluding rows
# where lag features are NaN (first ~49 days of data)
train_mask = (
    (df['is_train'] == True) &
    (df['date'] < VAL_START) &
    (df['sales_lag_16'].notna())
)

# Validation: the held-out 16-day period
val_mask = (
    (df['is_train'] == True) &
    (df['date'] >= VAL_START) &
    (df['date'] <= VAL_END)
)

# Test: the actual competition prediction period
test_mask = (df['is_train'] == False)

# Extract feature matrices and target vectors
X_train = df.loc[train_mask, feature_cols]
y_train = df.loc[train_mask, 'sales_log1p']

X_val = df.loc[val_mask, feature_cols]
y_val = df.loc[val_mask, 'sales_log1p']

X_test = df.loc[test_mask, feature_cols]
test_ids = df.loc[test_mask, 'id']

print(f"\n{'=' * 50}")
print(f"DATA SPLIT SUMMARY")
print(f"{'=' * 50}")
print(f"Training set:   {X_train.shape[0]:>12,} samples")
print(f"Validation set: {X_val.shape[0]:>12,} samples")
print(f"Test set:       {X_test.shape[0]:>12,} samples")
print(f"\nValidation period: {VAL_START.date()} to {VAL_END.date()}")
print(f"Test period:       {test['date'].min().date()} to {test['date'].max().date()}")

In [ ]:
# ============================================================
# 6.3 Train LightGBM Model
# ============================================================

# Hyperparameters tuned for this competition
params = {
    'objective': 'regression',       # Standard regression loss (MSE)
    'metric': 'rmse',               # RMSE on log1p target = RMSLE on original
    'boosting_type': 'gbdt',        # Gradient Boosted Decision Trees
    'learning_rate': 0.03,          # Conservative learning rate for better generalization
    'num_leaves': 127,              # Maximum leaves per tree (2^7 - 1)
    'max_depth': -1,                # No depth limit (controlled by num_leaves)
    'min_child_samples': 50,        # Minimum samples per leaf (reduces overfitting)
    'feature_fraction': 0.8,        # Random feature subsampling per tree
    'bagging_fraction': 0.8,        # Random row subsampling per tree
    'bagging_freq': 5,              # Apply bagging every 5 iterations
    'reg_alpha': 0.1,               # L1 regularization
    'reg_lambda': 0.1,              # L2 regularization
    'n_estimators': 2000,           # Maximum boosting rounds (early stopping will decide)
    'verbose': -1,                  # Suppress per-tree output
    'random_state': 42,             # Reproducibility seed
    'n_jobs': -1                    # Use all CPU cores
}

# Create LightGBM-native dataset objects for efficient training
dtrain = lgb.Dataset(X_train, label=y_train)
dval = lgb.Dataset(X_val, label=y_val, reference=dtrain)

# Training callbacks: log progress every 200 rounds, stop if no improvement for 100 rounds
callbacks = [
    lgb.log_evaluation(period=200),
    lgb.early_stopping(stopping_rounds=100)
]

# Train the model
print("Training LightGBM model...")
print("=" * 60)

model = lgb.train(
    params,
    dtrain,
    valid_sets=[dtrain, dval],
    valid_names=['train', 'validation'],
    callbacks=callbacks
)

print(f"\nTraining complete!")
print(f"Best iteration: {model.best_iteration}")
print(f"Best validation RMSE (log1p scale): {model.best_score['validation']['rmse']:.6f}")

## 📊 7. Model Evaluation

We evaluate the model on the held-out validation set using RMSLE — the official competition metric.

In [ ]:
# ============================================================
# 7.1 Calculate RMSLE on Validation Set
# ============================================================

def rmsle(y_true, y_pred):
    """
    Calculate Root Mean Squared Logarithmic Error.
    
    RMSLE = sqrt(mean((log(1 + y_true) - log(1 + y_pred))^2))
    
    Parameters:
        y_true: Actual values (original scale)
        y_pred: Predicted values (original scale)
    
    Returns:
        RMSLE score (lower is better)
    """
    # Clip predictions to non-negative values (sales cannot be negative)
    y_pred = np.clip(y_pred, 0, None)
    return np.sqrt(np.mean((np.log1p(y_true) - np.log1p(y_pred)) ** 2))

# Generate predictions on validation set (output is in log1p scale)
val_pred_log1p = model.predict(X_val, num_iteration=model.best_iteration)

# Convert predictions back to original sales scale
val_pred = np.expm1(val_pred_log1p)    # Inverse of log1p
val_pred = np.clip(val_pred, 0, None)  # Ensure non-negative predictions

# Get actual validation sales in original scale
val_actual = np.expm1(y_val.values)

# Calculate the official competition metric
val_rmsle = rmsle(val_actual, val_pred)

# Also calculate RMSE on log1p scale for cross-reference
val_rmse_log = np.sqrt(mean_squared_error(y_val, val_pred_log1p))

print("=" * 60)
print("MODEL EVALUATION RESULTS")
print("=" * 60)
print(f"\n  Validation RMSLE:              {val_rmsle:.6f}")
print(f"  Validation RMSE (log1p scale): {val_rmse_log:.6f}")
print(f"\n  Prediction Statistics (original scale):")
print(f"    {'Metric':<20s} {'Predicted':>12s} {'Actual':>12s}")
print(f"    {'-' * 45}")
print(f"    {'Mean':<20s} {val_pred.mean():>12.2f} {val_actual.mean():>12.2f}")
print(f"    {'Std':<20s} {val_pred.std():>12.2f} {val_actual.std():>12.2f}")
print(f"    {'Min':<20s} {val_pred.min():>12.2f} {val_actual.min():>12.2f}")
print(f"    {'Max':<20s} {val_pred.max():>12.2f} {val_actual.max():>12.2f}")

### 7.2 Feature Importance

In [ ]:
# ============================================================
# Visualize which features the model relies on most
# ============================================================

# Extract feature importance scores (using 'gain' metric)
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importance(importance_type='gain')
}).sort_values('importance', ascending=True)

# Plot top 25 most important features
TOP_N = 25
fig, ax = plt.subplots(figsize=(10, 10))
top_features = importance.tail(TOP_N)
colors_imp = plt.cm.RdYlGn(np.linspace(0.2, 0.9, TOP_N))

ax.barh(range(TOP_N), top_features['importance'].values, color=colors_imp)
ax.set_yticks(range(TOP_N))
ax.set_yticklabels(top_features['feature'].values, fontsize=10)
ax.set_xlabel('Feature Importance (Gain)', fontsize=12)
ax.set_title(f'Top {TOP_N} Most Important Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Print the top 10 feature importances numerically
print(f"\nTop 10 Features by Importance (Gain):")
for i, (_, row) in enumerate(importance.tail(10).iloc[::-1].iterrows(), 1):
    print(f"  {i:>2}. {row['feature']:<30s} {row['importance']:>12,.0f}")

### 7.3 Validation Predictions vs Actual Sales

In [ ]:
# ============================================================
# Compare predicted vs actual sales for sample store-family pairs
# ============================================================

# Select a few representative store-family pairs
sample_pairs = [
    (1, 'GROCERY I'),
    (3, 'BEVERAGES'),
    (44, 'PRODUCE'),
]

fig, axes = plt.subplots(len(sample_pairs), 1, figsize=(14, 4 * len(sample_pairs)))

# Create a DataFrame with validation predictions
val_df = df.loc[val_mask].copy()
val_df['predicted'] = val_pred

for idx, (store_id, family_name) in enumerate(sample_pairs):
    ax = axes[idx] if len(sample_pairs) > 1 else axes
    
    # Filter for this specific store-family pair
    pair_mask = (val_df['store_nbr'] == store_id) & (val_df['family'] == family_name)
    pair_data = val_df.loc[pair_mask]
    
    if len(pair_data) == 0:
        continue
    
    # Plot actual vs predicted
    ax.plot(pair_data['date'].values, np.expm1(pair_data['sales_log1p']).values,
            'o-', label='Actual', color='#2196F3', markersize=7, linewidth=2)
    ax.plot(pair_data['date'].values, pair_data['predicted'].values,
            's--', label='Predicted', color='#FF5722', markersize=7, linewidth=2)
    
    # Calculate per-pair RMSLE
    pair_rmsle = rmsle(np.expm1(pair_data['sales_log1p']).values, pair_data['predicted'].values)
    ax.set_title(f'Store {store_id} — {family_name}  |  RMSLE: {pair_rmsle:.4f}',
                 fontsize=12, fontweight='bold')
    ax.set_ylabel('Sales')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis='x', rotation=45)

plt.suptitle('Validation: Predicted vs Actual Sales', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 📤 8. Generate Submission

Generate predictions for the test period (Aug 16–31, 2017) and create the submission CSV file matching the required format.

In [ ]:
# ============================================================
# 8.1 Predict on Test Set
# ============================================================

# Generate predictions in log1p scale
test_pred_log1p = model.predict(X_test, num_iteration=model.best_iteration)

# Convert back to original sales scale
test_pred = np.expm1(test_pred_log1p)
test_pred = np.clip(test_pred, 0, None)  # Ensure non-negative sales

print("Test Prediction Statistics:")
print(f"  Min:    {test_pred.min():.4f}")
print(f"  Max:    {test_pred.max():.4f}")
print(f"  Mean:   {test_pred.mean():.4f}")
print(f"  Median: {np.median(test_pred):.4f}")
print(f"  Zeros:  {(test_pred == 0).sum()} ({(test_pred == 0).mean() * 100:.1f}%)")

# ============================================================
# 8.2 Create Submission File
# ============================================================

# Build submission DataFrame matching the required format: (id, sales)
submission = pd.DataFrame({
    'id': test_ids.astype(int).values,
    'sales': test_pred
})

# Sort by id to match the sample submission order
submission = submission.sort_values('id').reset_index(drop=True)

# Verify the submission format matches expectations
print(f"\nSubmission shape: {submission.shape} (expected: {sample_submission.shape})")
print(f"ID range: {submission['id'].min()} to {submission['id'].max()}")
assert submission.shape == sample_submission.shape, "Shape mismatch with sample submission!"
assert (submission['id'] == sample_submission['id']).all(), "ID mismatch with sample submission!"

# Save to CSV
submission.to_csv('submission.csv', index=False)
print(f"\nSubmission saved to 'submission.csv'")

# Display preview
print(f"\nSubmission Preview (first 10 rows):")
print(submission.head(10).to_string(index=False))

## 📋 Summary

### Approach
| Aspect | Details |
|--------|---------|
| **Model** | LightGBM (Gradient Boosted Decision Trees) |
| **Target** | `log1p(sales)` — transforms RMSLE into standard RMSE |
| **Features** | Calendar, lag (≥16d), rolling stats, oil prices, promotions, store metadata |
| **Validation** | Time-based split — last 16 days of training data |
| **Prediction** | `expm1()` inverse transform + clip to non-negative |

### Key Design Decisions
1. **Minimum lag = 16 days**: Matches the 16-day forecast horizon to prevent any data leakage
2. **Log1p transformation**: Converts the RMSLE objective to RMSE — LightGBM's native regression loss
3. **Rolling on shifted data**: Rolling windows are computed on pre-shifted columns (window ≤ shift) to guarantee group boundary safety
4. **Cyclical encoding**: Sine/cosine transforms for temporal features preserve their circular nature

### Potential Improvements
- **Ensemble**: Combine LightGBM with XGBoost, CatBoost, or Ridge Regression
- **Hyperparameter tuning**: Use Optuna or Bayesian optimization for systematic tuning
- **Additional features**: Store × family interactions, holiday proximity, earthquake impact features
- **Target encoding**: Mean-encoded features for high-cardinality categoricals
- **Recursive models**: Train separate models for different forecast horizons (day 1–8, day 9–16)